In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:

use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph import DFGrepInterference, DFGrepWorkflow 

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "deepspeedhigh" 

condition_fn = None #

if app_name == "mummi":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/mummi-32-node/*pfw.gz"

elif app_name == "montage":
    filename ="/usr/workspace/iopp/graph-io/dlp_logs/montage_16_48ppn/montage*.pfw"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m2d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-2-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m7d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-7-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "deepspeedlow":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_low_interference/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/deepspeed"
elif app_name == "deepspeedhigh":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_high_interference/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/deepspeed"
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, 
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=False, 
                                host_pattern=r'lassen(\d+)', time_granularity=30e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [20:58:57] Initialized Client with 128 workers and link http://134.9.71.28:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:673]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [20:59:02] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:665]


In [4]:

def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [5]:
def deepspeed_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])
    
    if "args" in json_object:
        if "fname" in json_object["args"]:
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))

    return d
load_cols_deepspeed = {'mount_point':"string[pyarrow]"}


In [6]:
analyzer_deepspeed = DFAnalyzer(filename,load_fn=deepspeed_cols_function, load_cols=load_cols_deepspeed, load_data={"mount_point":trie})

[INFO] [20:59:10] Created index for 15 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:371]
[INFO] [20:59:10] Total size of all files are <dask.bag.core.Item object at 0x1554ac13a6d0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:373]
[INFO] [20:59:12] Loading 8424 batches out of 15 files and has 137896085 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:386]
[INFO] [21:00:46] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:431]
[INFO] [21:00:46] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:437]


In [8]:
analyzer_deepspeed.events.cat.unique().compute()

0    dlio_benchmark
1           storage
2             POSIX
3            config
4      ai_framework
5        checkpoint
6             STDIO
7       data_loader
8            reader
Name: cat, dtype: string

In [10]:
eventsDF = analyzer_deepspeed.events.query('cat == "POSIX"')

In [11]:
eventsDF['id'] = eventsDF.index

In [14]:

# eventsDF = analyzer_montage.events[analyzer_montage.events['cat'] == "POSIX" ] # only posix events

In [12]:
cp_dir

'/p/lustre2/pandey2/cp_dir/deepspeed'

In [13]:
IFCalculator = DFGrepInterference(eventsDF, app_name=app_name, cp_dir=cp_dir, existing=False)

In [14]:
IFCalculator.get_degree()
IFCalculator.get_interference()
IFCalculator.get_interference_metadata()

RuntimeError: P2P shuffling b0c550dda1cd031b88511ae283fe7d62 failed during transfer phase

In [30]:
IFCalculator.deg_metadata.compute()

RuntimeError: shuffle_barrier failed during shuffle cc8832e200889c99506657bf1a8195d7

In [32]:
analyzer_deepspeed.events.query('cat == "POSIX"').compute()

FutureCancelledError: ('query-c6e7b56a21456f73c5aac1f2e4ac94ee', 12) cancelled for reason: scheduler-connection-lost.
Client lost the connection to the scheduler. Please check your connection and re-run your work.

In [17]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)

In [21]:
IFCalculator.deg_data.compute()

,id,name,pid,size,ts,te,mount_point,dur,trange,deg
0,10829,write,0,134217728,660590087,660950423,/p/lustre2,360336,22,1
1,13631,write,0,134217728,661006110,661369818,/p/lustre2,363708,22,1
2,105,write,0,134217728,661438391,661805701,/p/lustre2,367310,22,3
3,11235,write,12,134217728,661725733,662300105,/p/lustre2,574372,22,9
4,11014,write,15,134217728,661787472,662355067,/p/lustre2,567595,22,10
...,...,...,...,...,...,...,...,...,...,...
581,4492,write,0,134217728,1017327292,1017767067,/p/lustre2,439775,33,1
582,7360,write,0,134217728,1017850220,1018384076,/p/lustre2,533856,33,1
583,9923,write,0,134217728,1018467788,1018989776,/p/lustre2,521988,33,1
584,11272,write,0,134217728,1019068252,1022018865,/p/lustre2,2950613,33,1


In [23]:
metadata = IFCalculator.ddf_metadata.compute()

In [31]:
metadata

,id,name,pid,size,ts,te,mount_point,dur,trange
0,0,DLIOBenchmark.__init__,0,<NA>,56892,57161,<NA>,269,0
1,1,FileStorage.get_uri,0,<NA>,177252,177258,<NA>,6,0
2,2,opendir,0,<NA>,177303,220005,/p/lustre2,42702,0
3,3,FileStorage.walk_node,0,<NA>,177244,220094,<NA>,42850,0
4,4,FileStorage.get_uri,0,<NA>,220139,220144,<NA>,5,0
...,...,...,...,...,...,...,...,...,...
14566,14566,open64,9,<NA>,1236482155,1236482177,/p/lustre2,22,41
14567,14567,__fxstat64,9,<NA>,1236482191,1236482366,/p/lustre2,175,41
14568,14568,lseek64,9,<NA>,1236482379,1236482382,/p/lustre2,3,41
14569,14569,lseek64,9,<NA>,1236482393,1236482561,/p/lustre2,168,41


In [28]:
metadata.groupby('trange').count()

,id,name,pid,size,ts,te,mount_point,dur
trange,,,,,,,,
0,360,360,360,0,360,360,90,360
1,70,70,70,0,70,70,0,70
2,3735,3735,3735,0,3735,3735,3655,3735
3,120,120,120,0,120,120,0,120
4,1538423,1538423,1538423,0,1538423,1538423,1230149,1538423
5,3636834,3636834,3636834,0,3636834,3636834,2908343,3636834
6,3614977,3614977,3614977,0,3614977,3614977,2890826,3614977
7,3629761,3629761,3629761,0,3629761,3629761,2902711,3629761
8,3253767,3253767,3253767,0,3253767,3253767,2602053,3253767


In [26]:
metadata.groupby('mount_point').count()

,id,name,pid,size,ts,te,dur,trange
mount_point,,,,,,,,
,4,4,4,0,4,4,4,4
/l/ssd,14363,14363,14363,438,14363,14363,14363,14363
/p/lustre2,78760912,78760912,78760912,0,78760912,78760912,78760912,78760912


In [13]:
IFCalculator.ddf.compute()

,name,cat,pid,tid,ts,te,dur,tinterval,trange,hostname,compute_time,io_time,app_io_time,total_time,filename,phase,size,mount_point,id
0,DLIOBenchmark.__init__,dlio_benchmark,0,1740017,56892,57161,269,<NA>,0,corona173,<NA>,<NA>,<NA>,0,<NA>,0,<NA>,<NA>,0
1,FileStorage.get_uri,storage,0,1740017,177252,177258,6,<NA>,0,corona173,<NA>,<NA>,<NA>,0,<NA>,0,<NA>,<NA>,1
2,opendir,POSIX,0,1740017,177303,220005,42702,<NA>,0,corona173,<NA>,42702,<NA>,42702,/p/lustre2/haridev/dlio/scr/dataset/scr_megatr...,2,<NA>,/p/lustre2,2
3,FileStorage.walk_node,storage,0,1740017,177244,220094,42850,<NA>,0,corona173,<NA>,<NA>,<NA>,0,<NA>,0,<NA>,<NA>,3
4,FileStorage.get_uri,storage,0,1740017,220139,220144,5,<NA>,0,corona173,<NA>,<NA>,<NA>,0,<NA>,0,<NA>,<NA>,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14566,open64,POSIX,9,2005324,1236482155,1236482177,22,<NA>,41,corona174,<NA>,22,<NA>,22,/p/lustre2/haridev/dlio/scr/dataset/scr_megatr...,2,<NA>,/p/lustre2,14566
14567,__fxstat64,POSIX,9,2005324,1236482191,1236482366,175,<NA>,41,corona174,<NA>,175,<NA>,175,/p/lustre2/haridev/dlio/scr/dataset/scr_megatr...,2,<NA>,/p/lustre2,14567
14568,lseek64,POSIX,9,2005324,1236482379,1236482382,3,<NA>,41,corona174,<NA>,3,<NA>,3,/p/lustre2/haridev/dlio/scr/dataset/scr_megatr...,2,<NA>,/p/lustre2,14568
14569,lseek64,POSIX,9,2005324,1236482393,1236482561,168,<NA>,41,corona174,<NA>,168,<NA>,168,/p/lustre2/haridev/dlio/scr/dataset/scr_megatr...,2,<NA>,/p/lustre2,14569


In [12]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)
IFCalculator.write_checkpoint("inter_metadata", cp_dir=cp_dir)

In [13]:
IFCalculator.inter.compute()

,name,pid,size,ts,te,mount_point,dur,trange,deg_caller,deg_other,min_dur,interference
0,write,0,134217728,660590087,660950423,/p/lustre2,360336,22,1,1,335032,0.929777
1,write,0,134217728,661006110,661369818,/p/lustre2,363708,22,1,1,335032,0.921157
2,write,0,134217728,661438391,661805701,/p/lustre2,367310,22,3,1,335032,0.912123
3,write,12,134217728,661725733,662300105,/p/lustre2,574372,22,9,1,335032,0.583301
4,write,15,134217728,661787472,662355067,/p/lustre2,567595,22,10,1,335032,0.590266
...,...,...,...,...,...,...,...,...,...,...,...,...
580,write,0,134217728,1016927240,1017271769,/p/lustre2,344529,33,1,1,335032,0.972435
581,write,0,134217728,1017327292,1017767067,/p/lustre2,439775,33,1,1,335032,0.761826
582,write,0,134217728,1017850220,1018384076,/p/lustre2,533856,33,1,1,335032,0.62757
583,write,0,134217728,1018467788,1018989776,/p/lustre2,521988,33,1,1,335032,0.641839


In [14]:
x = IFCalculator.inter.compute()

In [16]:
x.groupby('mount_point').count()

,name,pid,size,ts,te,dur,trange,deg_caller,deg_other,min_dur,interference
mount_point,,,,,,,,,,,
/,8900,8900,8900,8900,8900,8900,8900,8900,8900,8900,8900
/p/lustre2,4696,4696,4696,4696,4696,4696,4696,4696,4696,4696,4696
